In [ ]:
# Notebook setup
from scripts.utils import setup_notebook, COLOR_B_AXIS, COLOR_C_AXIS, OKABE_ITO_CYCLE
PROJECT_ROOT, np, pd, plt, Path = setup_notebook()

# Additional imports
import glob
import re

In [ ]:
from scripts.utils import OKABE_ITO_CYCLE
color_arr = OKABE_ITO_CYCLE

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# File paths for the data files

cwd = os.getcwd()

# --- Set Dynamic Paths ---
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
RAW_DATA_DIR = os.path.join(BASE_DIR, 'data', 'SQUID_bulk_CrSBr', 'MvH')
PLOTS_DIR = os.path.join(BASE_DIR, 'output', 'SQUID_bulk_CrSBr')
plot_name = os.path.join(PLOTS_DIR,'SQUID_Bulk_MvsH')
RESULTS_DIR = os.path.join(BASE_DIR, 'output', 'SQUID_bulk_CrSBr', 'results')

# Ensure folders exist
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Raw data folder: {RAW_DATA_DIR}")
print(f"Figures folder: {PLOTS_DIR}")

folder = RAW_DATA_DIR

mass_oop_sample = 1E-4 #KG
density = 3690 #kg/m3

file_paths = {
    "a-axis": os.path.join(folder,r"CrSBr_Hmax_30000 Oe_IPhard_10K.rso.dat"),
    "b-axis": os.path.join(folder,r"CrSBr_Hmax_10000 Oe_IPeasy_10K.rso.dat"),
    "c-axis": os.path.join(folder,r"CrSBr_Hmax_30000 Oe_OOP_10K.rso.dat")
}


In [ ]:
class SQUID_MvH_data:
    def __init__(self, filepath, label):
        self.filepath = filepath
        self.label = label
        self.data = pd.read_csv(self.filepath, delimiter =',', skiprows=30)
        self.field = self.data['Field (Oe)'].to_numpy()/10000   #field in T
        self.moment = self.data['Long Moment (emu)'].to_numpy()/1000 #magnetic moment in Am2
        self.moment_err = self.data['Long Scan Std Dev'].to_numpy()/1000
        self.normalise_data()
        self.sample_mass = 1E-4 #0.1 mg
        self.sample_density = 1 #write in kg/m3
        self.M_S = (self.sat_moment/self.sample_mass)*self.sample_density  #A/M

    def normalise_data(self):
        MS = (np.max(self.moment) - np.min(self.moment)) / 2
        self.sat_moment = MS
        self.offset = (np.max(self.moment) + np.min(self.moment)) / 2
        self.M_norm = self.moment / MS
        self.Err_norm = self.moment_err / MS
        self.M_centered = self.moment - self.offset
        self.error_norm = self.moment_err / MS


In [ ]:
M_H_axis_a = SQUID_MvH_data(os.path.join(folder,r"CrSBr_Hmax_30000 Oe_IPhard_10K.rso.dat"), label='a-axis')
M_H_axis_b = SQUID_MvH_data(os.path.join(folder,r"CrSBr_Hmax_10000 Oe_IPeasy_10K.rso.dat"), label='b-axis')
M_H_axis_c = SQUID_MvH_data(os.path.join(folder,r"CrSBr_Hmax_30000 Oe_OOP_10K.rso.dat"), label='c-axis')

In [ ]:
# --- Assume you already have your MeasurementData objects ---
all_samples = [M_H_axis_a, M_H_axis_b, M_H_axis_c]

# --- Create figure ---
fig, ax = plt.subplots(figsize=(6, 5), dpi=600)

# --- Loop over samples and plot ---
for idx, sample in enumerate(all_samples):
    color = OKABE_ITO_CYCLE[idx % len(color_arr)]  # Cycle colors safely
    ax.errorbar(sample.field,sample.M_norm, sample.error_norm, fmt ='-o',label=sample.label,color=color, markersize=3)

# --- Labels and Styling ---
ax.set_xlabel(r"$H$ (T)")
ax.set_ylabel(r"Magnetization ($M/M_S$)")
ax.legend(loc='best', frameon=False)
ax.set_xticks(np.arange(-3, 4, 1))

# --- Tight Layout for clean spacing ---
plt.tight_layout()

# --- Save the figure (optional) ---
save_path = os.path.join(PLOTS_DIR, "CrSBr_all_axes_MvH_plot.png")
fig.savefig(save_path, dpi=600, bbox_inches='tight', transparent = True)

# --- Show the plot ---
plt.show()

print(f"Figure saved at {save_path}")



In [ ]:
# --- Create figure ---
fig, ax = plt.subplots(figsize=(6, 5), dpi=600)

# --- Loop over samples and plot ---
for idx, sample in enumerate(all_samples):
    color = color_arr[idx % len(color_arr)]  # Cycle colors safely
    ax.errorbar(sample.field, sample.M_norm, sample.error_norm, fmt='-o',
                label=sample.label, color=color, markersize=3)

# --- Create inset axes for the zoomed-in view ---
ax_inset = ax.inset_axes([0.6, 0.1, 0.4, 0.4])  # [x, y, width, height] in fraction of figure

# --- Plot only the b-axis data in the inset ---
b_axis_data = all_samples[1]  # Assuming b-axis is the second sample
ax_inset.errorbar(b_axis_data.field, b_axis_data.M_norm, b_axis_data.error_norm,
                 fmt='-o', color=color_arr[1], markersize=2, linewidth=1)

# --- Set limits for the inset to show -0.6T to 0.6T ---
ax_inset.set_xlim(-0.4, 0.4)
ax_inset.set_ylim(-0.2, 0.2)
ax_inset.set_xticks([-0.4, -0.2, 0, 0.2])
ax_inset.set_yticks([])
ax_inset.set_xticklabels(['-0.4', '-0.2', '0', '0.2'], fontsize=10)
ax_inset.tick_params(axis='both', which='major', labelsize=10)

# --- Add rectangle in main plot to show where the inset is zooming from ---
ax.indicate_inset_zoom(ax_inset, edgecolor="black", alpha=0.5)

# --- Labels and Styling for main plot ---
ax.set_xlabel(r"$H$ (T)")
ax.set_ylabel(r"Magnetization ($M/M_S$)")
ax.legend(loc='best', frameon=False)
ax.set_xticks(np.arange(-3, 4, 1))

# --- Tight Layout for clean spacing ---
plt.tight_layout()

# --- Save the figure (optional) ---
save_path = os.path.join(PLOTS_DIR, "CrSBr_all_axes_MvH_plot_with_inset.png")
fig.savefig(save_path, dpi=600, bbox_inches='tight', transparent=True)

# --- Show the plot ---
plt.show()

print(f"Figure saved at {save_path}")

In [ ]:
positive_field_subsets = {}
for sample in all_samples:
    positive_mask = (sample.field > 0) & (sample.field < 2)
    field_subset = sample.field[positive_mask]
    moment_subset = sample.M_norm[positive_mask]
    positive_field_subsets[sample.label] = {
        "Field (Oe)": field_subset * 10000,  # Convert back to Oe from Tesla
        "Long Moment (emu)": moment_subset * sample.sat_moment  # Convert from M/Ms back to emu
    }

In [ ]:
H_T_g = []
M_norm_g = []
area_g = []


for label, subset in positive_field_subsets.items():
    MS = (np.max(subset['Long Moment (emu)'])-np.min(subset['Long Moment (emu)']))
    M_norm = np.array(subset['Long Moment (emu)'])/MS
    H_T = np.array(subset['Field (Oe)'])/10000
    M_norm_g.append(M_norm)
    H_T_g.append(H_T)

    plt.plot(H_T, M_norm, label = label)

plt.title("Positive Field (Oe) vs Long Moment (emu) at 10K")
plt.xlabel("Field (T)")
plt.ylabel("Long Moment (emu)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

#Calculating area under the curve to calculate Integration H.dM

H_sat_a = [0.5,1.05,1.82]
i=0
while i<3 : #calculating for field sweep along a, b and c axis
    j=1
    area=0
    while j<len(H_T_g[i]) :
        area = area + np.abs(((H_T_g[i][j]+H_T_g[i][j-1])/2))*np.abs(((M_norm_g[i][j]-M_norm_g[i][j-1]))) #H_T_g[][] is field for a certain axis at a certain datapoint,  M_norm_g[][] is magnetization at for certain axis at a certain datapoint.
        j=j+1
    area_g.append(area/2)
    i=i+1


K_eff_a = area_g[0] #T (unit is Tesla because M was M/Ms)
K_eff_b = area_g[1] #T
K_eff_c = area_g[2] #T

K_eff_IP = area_g[0] - area_g[1] #T (unit is Tesla because M was M/Ms)
K_eff_OOP = area_g[2] - area_g[1] #T

print("K_eff_IP = ",K_eff_IP,"T")
print("K_eff_OOP = ",K_eff_OOP,"T")


In [ ]:
# --- Directory to save individual CSVs ---
save_folder = os.path.join(RESULTS_DIR, 'SQUID bulk')
os.makedirs(save_folder, exist_ok=True)  # Create folder if it doesn't exist

# --- Export each sample (field in T, moment normalised to M_S) ---
for sample in all_samples:
    export_df = pd.DataFrame({
        'Field (T)': sample.field,
        'Magnetization (M/MS)': sample.M_norm
    })

    # Clean filename: remove spaces and special characters
    safe_label = sample.label.replace(' ', '_').replace('/', '_')

    save_path = os.path.join(save_folder, f'{safe_label}_Normalized.csv')
    export_df.to_csv(save_path, index=False)

    print(f"Exported {sample.label} to {save_path}")
